# memory_model
**Prerequisites:** variables, data types, operators

**Outcome:** After this notebook you will know exactly what Python stores when you write `x = 5`, why two variables can silently share the same object, and how to prove it with code.

## The Problem

In [ ]:
a = [1, 2, 3]
b = a
b.append(4)

print(a)  # why did a change? we only touched b

## The Concept

Python never stores a value directly inside a variable name. A variable is a label. It points to an object that lives in memory. When you write `b = a`, you are not copying the list — you are making a second label that points to the same object. Both labels see the same list. Changing the list through one label changes what the other label sees, because there is only one list.

## Minimal Example

In [ ]:
x = 42
print(id(x))   # memory address of the object 42

y = x
print(id(y))   # same address — same object, two labels

## How Python Does It

Every Python object has three things: a type, a value, and a reference count. The reference count tracks how many labels point to it. When the count drops to zero, the object is eligible for garbage collection. `id()` returns the memory address of the object for the duration of its lifetime. `is` compares addresses, `==` compares values.

In [ ]:
import sys

jobs = ["extract", "transform", "load"]

print("type  :", type(jobs))
print("id    :", id(jobs))
print("refs  :", sys.getrefcount(jobs))  # +1 because getrefcount itself holds a ref

## Building Up

In [ ]:
# assignment binds a name to an object — it does not copy
a = ["api", "worker"]
b = a

print(id(a) == id(b))  # True — same object
print(a is b)          # True — is checks identity

In [ ]:
# rebinding a name does not affect the other name
a = ["api", "worker"]
b = a
a = ["scheduler"]      # a now points to a new object

print(a)   # ["scheduler"]
print(b)   # ["api", "worker"] — b still points to the original

In [ ]:
# mutating through one name is visible through all names pointing to the same object
a = ["api", "worker"]
b = a
b.append("scheduler")

print(a)   # ["api", "worker", "scheduler"]
print(b)   # ["api", "worker", "scheduler"]

In [ ]:
# integers behave the same way — but small integers are cached (interned)
x = 256
y = 256
print(x is y)   # True — CPython caches -5 to 256

x = 257
y = 257
print(x is y)   # False — outside the cache range, two separate objects

In [ ]:
# function arguments pass references, not copies
def add_service(services, name):
    services.append(name)

running = ["api", "worker"]
add_service(running, "scheduler")
print(running)   # ["api", "worker", "scheduler"] — original modified

## Where It Breaks

In [ ]:
# silent bug — two variables share state without the programmer realising
config_a = {"retries": 3, "timeout": 30}
config_b = config_a
config_b["timeout"] = 99

print(config_a)   # {"retries": 3, "timeout": 99} — config_a changed silently

In [ ]:
# using is to compare values — works accidentally for small ints, fails elsewhere
a = 1000
b = 1000
print(a == b)   # True  — correct: compares values
print(a is b)   # False — wrong tool: compares identity, not value

## The Fix

In [ ]:
# use .copy() to get an independent shallow copy
config_a = {"retries": 3, "timeout": 30}
config_b = config_a.copy()
config_b["timeout"] = 99

print(config_a)   # {"retries": 3, "timeout": 30} — unchanged
print(config_b)   # {"retries": 3, "timeout": 99}

In [ ]:
# use == to compare values, is only for None/True/False
status = None
print(status is None)    # correct use of is
print(status == None)    # works but is not idiomatic

## In a Real System

In [ ]:
# a pipeline stage receives a job record and must not mutate the original
def enrich_record(record):
    enriched = record.copy()          # work on a copy, not the original
    enriched["processed"] = True
    enriched["source"] = "pipeline"
    return enriched

original = {"id": "job_1", "status": "pending"}
result = enrich_record(original)

print(original)   # unchanged — safe to retry or audit
print(result)     # enriched version

## Performance

Reference passing is O(1) regardless of object size. Copying is O(n). In tight loops over large collections, passing references is the default Python behaviour and is usually what you want. The cost appears when you accidentally share mutable state — not in the reference passing itself, but in the debugging hours it produces.

## Anti-Pattern

In [ ]:
# anti-pattern: assuming assignment makes a copy
batch_1 = ["job_1", "job_2"]
batch_2 = batch_1           # not a copy — same object
batch_2.append("job_3")
print(batch_1)              # ["job_1", "job_2", "job_3"] — surprise

# correct
batch_1 = ["job_1", "job_2"]
batch_2 = batch_1.copy()    # independent copy
batch_2.append("job_3")
print(batch_1)              # ["job_1", "job_2"] — untouched

## Interview Signals

- What is the difference between `is` and `==` in Python?
- Why does assigning a list to a new variable not create a copy?
- What does `id()` return and when does it change?
- Why are small integers like `256` the same object but `257` is not?

## Exercise

In [ ]:
def process_jobs(job_list):
    """
    Bug: this function is supposed to return a new list
    with 'processed_' prefixed to each job id.
    It must NOT modify the original list.
    Currently it has a memory model bug — find it and fix it.
    """
    result = job_list
    for i in range(len(result)):
        result[i] = "processed_" + result[i]
    return result


original = ["job_1", "job_2", "job_3"]
processed = process_jobs(original)

assert processed == ["processed_job_1", "processed_job_2", "processed_job_3"]
assert original == ["job_1", "job_2", "job_3"], "original must not be modified"
print("all assertions passed")